# CardioVis — segment videos and merge by surgical stage

This notebook:

1. Reads the large source videos directly from mounted Google Drive.
2. Cuts the timestamp intervals from the provided stage document.
3. Normalizes each short clip to 1080p/30 fps for reliable concatenation.
4. Creates:
   - one merged video per **procedure + stage**;
   - one merged video per **stage across both procedures**;
   - CSV reports for completed, missing, and ambiguous cases.

**Important:** The first run is a dry run. It checks paths without encoding.  
After reviewing `segment_report.csv`, change `DRY_RUN = True` to `DRY_RUN = False` in the pipeline cell and run it again.

Rows marked **Gốc/original** require the original-video folder, not only `CardioVis-merged-videos`.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import shutil, subprocess
if shutil.which("ffmpeg") is None:
    subprocess.run(["apt-get", "update", "-qq"], check=True)
    subprocess.run(["apt-get", "install", "-y", "-qq", "ffmpeg"], check=True)

print("Drive mounted and FFmpeg is ready.")

## Legacy pipeline (deprecated)

Do not run the next legacy cell unless you intentionally need the old workflow.

Use the new section below: **Run the new Drive pipeline (FPS=20 + sample 1/20)**.

The legacy cell uses `ORIGINAL_DIR` and is kept only for reference.

In [ ]:

from __future__ import annotations

import os

if os.getenv('ALLOW_LEGACY_PIPELINE', '0') != '1':
    raise RuntimeError(
        'This is the legacy pipeline cell and is disabled by default. '
        'Run the new runner cell under "Run the new Drive pipeline (FPS=20 + sample 1/20)".'
    )

import csv
import hashlib
import re
import shutil
import subprocess
import unicodedata
from collections import defaultdict
from pathlib import Path
from typing import Any, Optional

# ============================================================
# 1) EDIT ONLY THESE PATHS IF NEEDED
# ============================================================
MERGED_DIR = Path("/content/drive/MyDrive/CardioVis-merged-videos")

# Rows marked "Gốc/original" in the document must be read from here.
# Change this to the actual folder containing the original per-patient videos.
ORIGINAL_DIR = Path("/content/drive/MyDrive/CardioVis-original-videos")

OUTPUT_DIR = Path("/content/drive/MyDrive/CardioVis-stage-segments")

# Local Colab scratch space. Final outputs are copied to Drive safely.
WORK_DIR = Path("/content/CardioVis_stage_work")

# Re-run behavior
OVERWRITE_SEGMENTS = False
OVERWRITE_MERGES = False

# Set True for a fast path/timestamp check without cutting or merging.
DRY_RUN = True

# Standardize every short clip so all clips can be concatenated reliably.
TARGET_WIDTH = 1920
TARGET_HEIGHT = 1080
TARGET_FPS = 30
CRF = 18
PRESET = "veryfast"

# Optional exact paths for original videos when automatic search is ambiguous.
# Key format: ("replacement" or "repair", patient_number, video_number)
ORIGINAL_OVERRIDES: dict[tuple[str, int, int], str] = {
    # Example:
    # ("replacement", 7, 2): "/content/drive/MyDrive/.../Patient_7/video2.mp4",
}

# Optional exact paths for merged videos if a filename differs from the screenshot.
# Key format: ("replacement" or "repair", patient_number)
MERGED_OVERRIDES: dict[tuple[str, int], str] = {
    # Example:
    # ("repair", 10): "/content/drive/MyDrive/.../Valve_Repair_Patient_10_merged.mp4",
}

# ============================================================
# 2) TIMESTAMP MANIFEST TRANSCRIBED FROM passio-lab-1.docx
#
# Time notation:
#   "36.51"    -> 00:36:51  (minutes.seconds)
#   "1.30.06"  -> 01:30:06  (hours.minutes.seconds)
# ============================================================
MANIFEST: list[dict[str, Any]] = [
    # -------------------- VALVE REPLACEMENT --------------------
    {"procedure":"replacement","patient":1,"stage":"root_needle","source":"merged","video_no":None,"start":"36.51","end":"37.06"},
    {"procedure":"replacement","patient":1,"stage":"atriotomy","source":"merged","video_no":None,"start":"56.38","end":"56.53"},

    {"procedure":"replacement","patient":2,"stage":"root_needle","source":"merged","video_no":None,"start":"16.41","end":"16.56"},
    {"procedure":"replacement","patient":2,"stage":"atriotomy","source":"merged","video_no":None,"start":"23.22","end":"23.37"},

    {"procedure":"replacement","patient":4,"stage":"root_needle","source":"merged","video_no":None,"start":"12.45","end":"13.00"},
    {"procedure":"replacement","patient":4,"stage":"atriotomy","source":"merged","video_no":None,"start":"20.02","end":"20.12"},

    {"procedure":"replacement","patient":5,"stage":"root_needle","source":"merged","video_no":None,"start":"7.46","end":"8.01"},
    {"procedure":"replacement","patient":5,"stage":"atriotomy","source":"merged","video_no":None,"start":"14.33","end":"14.43"},

    {"procedure":"replacement","patient":6,"stage":"root_needle","source":"merged","video_no":None,"start":"10.30","end":"10.43"},
    {"procedure":"replacement","patient":6,"stage":"atriotomy","source":"merged","video_no":None,"start":"19.07","end":"19.22"},

    {"procedure":"replacement","patient":7,"stage":"root_needle","source":"original","video_no":2,"start":"0.02","end":"0.12"},
    {"procedure":"replacement","patient":7,"stage":"atriotomy","source":"original","video_no":2,"start":"6.26","end":"6.36"},
    {"procedure":"replacement","patient":7,"stage":"phrenic_nerve","source":"original","video_no":1,"start":"4.42","end":"4.52"},

    {"procedure":"replacement","patient":8,"stage":"root_needle","source":"merged","video_no":None,"start":"11.33","end":"11.48"},
    {"procedure":"replacement","patient":8,"stage":"atriotomy","source":"merged","video_no":None,"start":"19.57","end":"20.11"},

    # ----------------------- VALVE REPAIR -----------------------
    {"procedure":"repair","patient":9,"stage":"root_needle","source":"original","video_no":3,"start":"0","end":"0.13"},
    {"procedure":"repair","patient":9,"stage":"atriotomy","source":"original","video_no":3,"start":"9.06","end":"9.13"},

    {"procedure":"repair","patient":10,"stage":"root_needle","source":"merged","video_no":None,"start":"10.35","end":"10.41"},
    {"procedure":"repair","patient":10,"stage":"atriotomy","source":"merged","video_no":None,"start":"17.40","end":"17.48"},
    {"procedure":"repair","patient":10,"stage":"phrenic_nerve","source":"merged","video_no":None,"start":"1.28","end":"1.35"},

    {"procedure":"repair","patient":11,"stage":"root_needle","source":"merged","video_no":None,"start":"23.15","end":"23.30"},
    {"procedure":"repair","patient":11,"stage":"atriotomy","source":"merged","video_no":None,"start":"46.08","end":"46.18"},
    {"procedure":"repair","patient":11,"stage":"phrenic_nerve","source":"merged","video_no":None,"start":"6.43","end":"6.49"},

    {"procedure":"repair","patient":12,"stage":"root_needle","source":"merged","video_no":None,"start":"9.16","end":"9.30"},
    {"procedure":"repair","patient":12,"stage":"atriotomy","source":"merged","video_no":None,"start":"22.36","end":"22.46"},
    {"procedure":"repair","patient":12,"stage":"phrenic_nerve","source":"merged","video_no":None,"start":"2.26","end":"2.32"},

    {"procedure":"repair","patient":13,"stage":"root_needle","source":"original","video_no":1,"start":"0.05","end":"0.21"},
    {"procedure":"repair","patient":13,"stage":"atriotomy","source":"original","video_no":2,"start":"0.15","end":"0.24"},
    {"procedure":"repair","patient":13,"stage":"phrenic_nerve","source":"original","video_no":9,"start":"0.02","end":"0.12"},

    {"procedure":"repair","patient":14,"stage":"root_needle","source":"merged","video_no":None,"start":"14.06","end":"14.21"},
    {"procedure":"repair","patient":14,"stage":"atriotomy","source":"merged","video_no":None,"start":"21.15","end":"21.25"},
    {"procedure":"repair","patient":14,"stage":"phrenic_nerve","source":"merged","video_no":None,"start":"7.07","end":"7.17"},

    {"procedure":"repair","patient":16,"stage":"root_needle","source":"merged","video_no":None,"start":"19.45","end":"19.58"},
    {"procedure":"repair","patient":16,"stage":"atriotomy","source":"merged","video_no":None,"start":"29.48","end":"30.01"},
    {"procedure":"repair","patient":16,"stage":"phrenic_nerve","source":"merged","video_no":None,"start":"2.25","end":"2.31"},

    {"procedure":"repair","patient":17,"stage":"root_needle","source":"original","video_no":14,"start":"0.27","end":"0.38"},

    {"procedure":"repair","patient":18,"stage":"root_needle","source":"merged","video_no":None,"start":"12.25","end":"12.37"},
    {"procedure":"repair","patient":18,"stage":"atriotomy","source":"merged","video_no":None,"start":"27.08","end":"27.23"},
    {"procedure":"repair","patient":18,"stage":"phrenic_nerve","source":"merged","video_no":None,"start":"3.18","end":"3.31"},

    {"procedure":"repair","patient":19,"stage":"root_needle","source":"merged","video_no":None,"start":"0.17","end":"0.32"},
    {"procedure":"repair","patient":19,"stage":"atriotomy","source":"merged","video_no":None,"start":"12.50","end":"13.04"},
    {"procedure":"repair","patient":19,"stage":"phrenic_nerve","source":"merged","video_no":None,"start":"2.25","end":"2.32"},

    {"procedure":"repair","patient":20,"stage":"root_needle","source":"original","video_no":2,"start":"0","end":"0.10"},
    {"procedure":"repair","patient":20,"stage":"atriotomy","source":"original","video_no":2,"start":"4.50","end":"5.00"},
    {"procedure":"repair","patient":20,"stage":"phrenic_nerve","source":"merged","video_no":None,"start":"0.15","end":"0.30"},

    {"procedure":"repair","patient":21,"stage":"root_needle","source":"merged","video_no":None,"start":"11.05","end":"11.20"},
    {"procedure":"repair","patient":21,"stage":"atriotomy","source":"merged","video_no":None,"start":"23.45","end":"24.00"},

    {"procedure":"repair","patient":22,"stage":"root_needle","source":"original","video_no":6,"start":"0","end":"0.10"},

    {"procedure":"repair","patient":23,"stage":"root_needle","source":"merged","video_no":None,"start":"10.56","end":"11.06"},
    {"procedure":"repair","patient":23,"stage":"atriotomy","source":"merged","video_no":None,"start":"22.09","end":"22.18"},
    {"procedure":"repair","patient":23,"stage":"phrenic_nerve","source":"merged","video_no":None,"start":"2.12","end":"2.18"},

    {"procedure":"repair","patient":24,"stage":"root_needle","source":"merged","video_no":None,"start":"13.09","end":"13.16"},
    {"procedure":"repair","patient":24,"stage":"atriotomy","source":"merged","video_no":None,"start":"22.03","end":"22.11"},
    {"procedure":"repair","patient":24,"stage":"phrenic_nerve","source":"merged","video_no":None,"start":"4.00","end":"4.08"},

    {"procedure":"repair","patient":26,"stage":"root_needle","source":"merged","video_no":None,"start":"13.53","end":"14.08"},
    {"procedure":"repair","patient":26,"stage":"atriotomy","source":"merged","video_no":None,"start":"23.07","end":"23.22"},
    {"procedure":"repair","patient":26,"stage":"phrenic_nerve","source":"merged","video_no":None,"start":"2.25","end":"2.35"},

    {"procedure":"repair","patient":27,"stage":"root_needle","source":"merged","video_no":None,"start":"29.08","end":"29.20"},
    {"procedure":"repair","patient":27,"stage":"atriotomy","source":"merged","video_no":None,"start":"40.47","end":"41.02"},
    {"procedure":"repair","patient":27,"stage":"phrenic_nerve","source":"merged","video_no":None,"start":"7.33","end":"7.42"},

    # Patient 28 root is missing and atriotomy has an invalid timestamp.
    {"procedure":"repair","patient":28,"stage":"phrenic_nerve","source":"merged","video_no":None,"start":"1.30.06","end":"1.30.16"},

    {"procedure":"repair","patient":29,"stage":"root_needle","source":"merged","video_no":None,"start":"1.11","end":"1.26"},
    {"procedure":"repair","patient":29,"stage":"atriotomy","source":"merged","video_no":None,"start":"11.14","end":"11.24"},
    {"procedure":"repair","patient":29,"stage":"phrenic_nerve","source":"merged","video_no":None,"start":"6.35","end":"6.45"},

    {"procedure":"repair","patient":30,"stage":"root_needle","source":"original","video_no":4,"start":"4.50","end":"5.00"},
    {"procedure":"repair","patient":30,"stage":"atriotomy","source":"original","video_no":4,"start":"6.22","end":"6.33"},
    {"procedure":"repair","patient":30,"stage":"phrenic_nerve","source":"original","video_no":1,"start":"2.11","end":"2.23"},

    {"procedure":"repair","patient":33,"stage":"atriotomy","source":"original","video_no":1,"start":"5.00","end":"5.09"},
    {"procedure":"repair","patient":33,"stage":"phrenic_nerve","source":"original","video_no":9,"start":"0.52","end":"1.00"},
]

REVIEW_REQUIRED: list[dict[str, Any]] = [
    {"procedure":"replacement","patient":1,"stage":"phrenic_nerve","issue":"No timestamp in document"},
    {"procedure":"replacement","patient":2,"stage":"phrenic_nerve","issue":"No timestamp in document"},
    {"procedure":"replacement","patient":3,"stage":"all","issue":"Beginning missing; no usable timestamps"},
    {"procedure":"replacement","patient":4,"stage":"phrenic_nerve","issue":"No timestamp in document"},
    {"procedure":"replacement","patient":5,"stage":"phrenic_nerve","issue":"No timestamp in document"},
    {"procedure":"replacement","patient":6,"stage":"phrenic_nerve","issue":"No timestamp in document"},
    {"procedure":"replacement","patient":8,"stage":"phrenic_nerve","issue":"No timestamp in document"},

    {"procedure":"repair","patient":9,"stage":"phrenic_nerve","issue":"Document says not shown"},
    {"procedure":"repair","patient":15,"stage":"all","issue":"Beginning missing; no usable timestamps"},
    {"procedure":"repair","patient":17,"stage":"atriotomy","issue":"No timestamp in document"},
    {"procedure":"repair","patient":17,"stage":"phrenic_nerve","issue":"No timestamp in document"},
    {"procedure":"repair","patient":21,"stage":"phrenic_nerve","issue":"Timestamp 2.32-2.42 is marked original, but original video number is missing"},
    {"procedure":"repair","patient":22,"stage":"atriotomy","issue":"No timestamp in document"},
    {"procedure":"repair","patient":22,"stage":"phrenic_nerve","issue":"No timestamp in document"},
    {"procedure":"repair","patient":28,"stage":"root_needle","issue":"No timestamp in document"},
    {"procedure":"repair","patient":28,"stage":"atriotomy","issue":"Document says 4.02-1.16; end is earlier than start, so it is not safe to cut"},
    {"procedure":"repair","patient":31,"stage":"all","issue":"Video number and timestamps are blank"},
    {"procedure":"repair","patient":32,"stage":"all","issue":"Video number and timestamps are blank"},
    {"procedure":"repair","patient":33,"stage":"root_needle","issue":"No timestamp in document"},
]

STAGE_SLUG = {
    "root_needle": "01_dat_kim_goc",
    "atriotomy": "02_rach_nhi",
    "phrenic_nerve": "03_than_kinh_hoanh",
}
PROCEDURE_ORDER = {"replacement": 0, "repair": 1}
STAGE_ORDER = {"root_needle": 0, "atriotomy": 1, "phrenic_nerve": 2}
VIDEO_EXTENSIONS = {".mp4", ".mov", ".mkv", ".m4v", ".avi", ".mts", ".m2ts"}

# ============================================================
# 3) HELPERS
# ============================================================
def run_command(cmd: list[str]) -> subprocess.CompletedProcess[str]:
    return subprocess.run(
        cmd,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
        check=False,
    )

def ensure_ffmpeg() -> None:
    for binary in ("ffmpeg", "ffprobe"):
        if shutil.which(binary) is None:
            raise RuntimeError(
                f"{binary} is not installed. In Colab run: !apt-get update -qq && !apt-get install -y -qq ffmpeg"
            )

def parse_doc_time(value: str) -> float:
    """
    Interpret document notation:
      SS        -> seconds
      MM.SS     -> minutes:seconds
      HH.MM.SS  -> hours:minutes:seconds
    """
    value = str(value).strip()
    if not value:
        raise ValueError("Empty timestamp")

    parts = value.split(".")
    if not all(part.isdigit() for part in parts):
        raise ValueError(f"Timestamp contains non-digits: {value}")

    nums = [int(part) for part in parts]

    if len(nums) == 1:
        total = nums[0]
    elif len(nums) == 2:
        minutes, seconds = nums
        if seconds >= 60:
            raise ValueError(f"Invalid seconds in timestamp: {value}")
        total = minutes * 60 + seconds
    elif len(nums) == 3:
        hours, minutes, seconds = nums
        if minutes >= 60 or seconds >= 60:
            raise ValueError(f"Invalid hh.mm.ss timestamp: {value}")
        total = hours * 3600 + minutes * 60 + seconds
    else:
        raise ValueError(f"Unsupported timestamp format: {value}")

    return float(total)

def seconds_to_hhmmss(seconds: float) -> str:
    total = int(round(seconds))
    hours = total // 3600
    minutes = (total % 3600) // 60
    secs = total % 60
    return f"{hours:02d}:{minutes:02d}:{secs:02d}"

def time_slug(value: str) -> str:
    sec = int(parse_doc_time(value))
    hours = sec // 3600
    minutes = (sec % 3600) // 60
    seconds = sec % 60
    return f"{hours:02d}h{minutes:02d}m{seconds:02d}s"

def normalized_text(value: str) -> str:
    value = unicodedata.normalize("NFKD", value)
    value = "".join(ch for ch in value if not unicodedata.combining(ch))
    return value.lower()

def video_files(root: Path) -> list[Path]:
    if not root.exists():
        return []
    return sorted(
        p for p in root.rglob("*")
        if p.is_file() and p.suffix.lower() in VIDEO_EXTENSIONS
    )

def patient_matches(path: Path, patient: int) -> bool:
    text = normalized_text(str(path))
    patterns = [
        rf"(?:patient|benh[\W_]*nhan)[\W_]*0*{patient}(?!\d)",
        rf"(?:^|[\W_])p[\W_]*0*{patient}(?!\d)",
    ]
    return any(re.search(pattern, text) for pattern in patterns)

def video_number_matches(path: Path, video_no: int) -> bool:
    text = normalized_text(str(path))
    stem = normalized_text(path.stem)
    patterns = [
        rf"(?:video|vid|clip)[\W_]*0*{video_no}(?!\d)",
    ]
    if any(re.search(pattern, text) for pattern in patterns):
        return True
    compact_stem = re.sub(r"[^a-z0-9]+", "", stem)
    return compact_stem in {
        str(video_no),
        f"video{video_no}",
        f"vid{video_no}",
        f"clip{video_no}",
    }

def procedure_matches(path: Path, procedure: str) -> bool:
    text = normalized_text(str(path))
    if procedure == "replacement":
        return "replacement" in text or "valvereplacement" in re.sub(r"[^a-z0-9]+", "", text)
    if procedure == "repair":
        return "repair" in text or "valverepair" in re.sub(r"[^a-z0-9]+", "", text)
    return False

def resolve_merged_file(
    procedure: str,
    patient: int,
    merged_index: list[Path],
) -> tuple[Optional[Path], str]:
    override = MERGED_OVERRIDES.get((procedure, patient))
    if override:
        path = Path(override)
        if path.exists():
            return path, "merged override"
        return None, f"Merged override does not exist: {path}"

    expected_names = []
    if procedure == "replacement":
        expected_names = [
            f"Valve_replacement_Patient_{patient}_merged.mp4",
            f"Valve_Replacement_Patient_{patient}_merged.mp4",
        ]
    else:
        expected_names = [
            f"Valve_Repair_Patient_{patient}_merged.mp4",
            f"Valve_repair_Patient_{patient}_merged.mp4",
        ]

    for name in expected_names:
        candidate = MERGED_DIR / name
        if candidate.exists():
            return candidate, "exact merged filename"

    candidates = [
        p for p in merged_index
        if patient_matches(p, patient)
        and procedure_matches(p, procedure)
        and "merged" in normalized_text(p.name)
    ]

    if len(candidates) == 1:
        return candidates[0], "auto-detected merged file"
    if len(candidates) == 0:
        return None, (
            f"No merged file found for procedure={procedure}, patient={patient} "
            f"under {MERGED_DIR}"
        )
    return None, (
        "Multiple merged candidates found; add MERGED_OVERRIDES entry: "
        + " | ".join(str(p) for p in candidates[:10])
    )

def resolve_original_file(
    procedure: str,
    patient: int,
    video_no: int,
    original_index: list[Path],
) -> tuple[Optional[Path], str]:
    override = ORIGINAL_OVERRIDES.get((procedure, patient, video_no))
    if override:
        path = Path(override)
        if path.exists():
            return path, "original override"
        return None, f"Original override does not exist: {path}"

    candidates = [
        p for p in original_index
        if patient_matches(p, patient) and video_number_matches(p, video_no)
    ]

    # Prefer candidates whose path also explicitly names the procedure.
    procedure_candidates = [p for p in candidates if procedure_matches(p, procedure)]
    if procedure_candidates:
        candidates = procedure_candidates

    if len(candidates) == 1:
        return candidates[0], "auto-detected original file"
    if len(candidates) == 0:
        return None, (
            f"No original file found for procedure={procedure}, patient={patient}, "
            f"video={video_no} under {ORIGINAL_DIR}. "
            "Set ORIGINAL_DIR correctly or add ORIGINAL_OVERRIDES."
        )
    return None, (
        "Multiple original candidates found; add ORIGINAL_OVERRIDES entry: "
        + " | ".join(str(p) for p in candidates[:10])
    )

def probe_duration(path: Path) -> Optional[float]:
    result = run_command([
        "ffprobe", "-v", "error",
        "-show_entries", "format=duration",
        "-of", "default=noprint_wrappers=1:nokey=1",
        str(path),
    ])
    if result.returncode != 0:
        return None
    try:
        return float(result.stdout.strip())
    except ValueError:
        return None

def valid_existing_video(path: Path, min_bytes: int = 10_000) -> bool:
    return path.exists() and path.is_file() and path.stat().st_size >= min_bytes

def safe_copy_to_drive(local_file: Path, drive_file: Path) -> None:
    drive_file.parent.mkdir(parents=True, exist_ok=True)
    partial = drive_file.with_name(drive_file.name + ".partial")
    if partial.exists():
        partial.unlink()
    shutil.copy2(local_file, partial)
    os.replace(partial, drive_file)

def cut_segment(
    source_path: Path,
    output_path: Path,
    start_seconds: float,
    end_seconds: float,
) -> tuple[bool, str]:
    duration = end_seconds - start_seconds
    if duration <= 0:
        return False, f"Invalid interval: end <= start ({start_seconds} -> {end_seconds})"

    source_duration = probe_duration(source_path)
    if source_duration is not None:
        if start_seconds >= source_duration:
            return False, (
                f"Start {seconds_to_hhmmss(start_seconds)} is beyond source duration "
                f"{seconds_to_hhmmss(source_duration)}"
            )
        if end_seconds > source_duration + 0.5:
            return False, (
                f"End {seconds_to_hhmmss(end_seconds)} is beyond source duration "
                f"{seconds_to_hhmmss(source_duration)}"
            )

    digest = hashlib.md5(str(output_path).encode("utf-8")).hexdigest()[:10]
    local_output = WORK_DIR / "segments_tmp" / f"{output_path.stem}_{digest}.mp4"
    local_output.parent.mkdir(parents=True, exist_ok=True)
    if local_output.exists():
        local_output.unlink()

    vf = (
        f"scale={TARGET_WIDTH}:{TARGET_HEIGHT}:force_original_aspect_ratio=decrease,"
        f"pad={TARGET_WIDTH}:{TARGET_HEIGHT}:(ow-iw)/2:(oh-ih)/2:color=black,"
        f"fps={TARGET_FPS},format=yuv420p"
    )

    cmd = [
        "ffmpeg",
        "-hide_banner", "-loglevel", "error", "-y",
        "-ss", f"{start_seconds:.3f}",
        "-i", str(source_path),
        "-t", f"{duration:.3f}",
        "-map", "0:v:0",
        "-vf", vf,
        "-an",
        "-c:v", "libx264",
        "-preset", PRESET,
        "-crf", str(CRF),
        "-profile:v", "high",
        "-level", "4.1",
        "-g", str(TARGET_FPS * 2),
        "-keyint_min", str(TARGET_FPS * 2),
        "-sc_threshold", "0",
        "-movflags", "+faststart",
        str(local_output),
    ]

    result = run_command(cmd)
    if result.returncode != 0 or not valid_existing_video(local_output):
        return False, f"ffmpeg failed: {result.stderr[-1500:]}"

    safe_copy_to_drive(local_output, output_path)
    local_output.unlink(missing_ok=True)
    return True, "created"

def ffconcat_quote(path: Path) -> str:
    # ffmpeg concat demuxer single-quote escaping
    return str(path).replace("'", r"'\''")

def concatenate_clips(
    clip_paths: list[Path],
    output_path: Path,
    overwrite: bool,
) -> tuple[bool, str]:
    clip_paths = [p for p in clip_paths if valid_existing_video(p)]
    if not clip_paths:
        return False, "No valid clips to merge"

    if valid_existing_video(output_path) and not overwrite:
        return True, "existing"

    merge_id = hashlib.md5(str(output_path).encode("utf-8")).hexdigest()[:10]
    concat_dir = WORK_DIR / "concat_tmp"
    concat_dir.mkdir(parents=True, exist_ok=True)
    list_file = concat_dir / f"{output_path.stem}_{merge_id}.txt"
    local_output = concat_dir / f"{output_path.stem}_{merge_id}.mp4"

    with list_file.open("w", encoding="utf-8") as handle:
        for clip in clip_paths:
            handle.write(f"file '{ffconcat_quote(clip.resolve())}'\n")

    if local_output.exists():
        local_output.unlink()

    # Fast path: every segment was normalized by the same encoder settings.
    copy_cmd = [
        "ffmpeg",
        "-hide_banner", "-loglevel", "error", "-y",
        "-f", "concat", "-safe", "0",
        "-i", str(list_file),
        "-fflags", "+genpts",
        "-avoid_negative_ts", "make_zero",
        "-c", "copy",
        "-movflags", "+faststart",
        str(local_output),
    ]
    result = run_command(copy_cmd)

    # Fallback: re-encode if stream-copy concatenation fails.
    if result.returncode != 0 or not valid_existing_video(local_output):
        local_output.unlink(missing_ok=True)
        encode_cmd = [
            "ffmpeg",
            "-hide_banner", "-loglevel", "error", "-y",
            "-f", "concat", "-safe", "0",
            "-i", str(list_file),
            "-an",
            "-c:v", "libx264",
            "-preset", PRESET,
            "-crf", str(CRF),
            "-pix_fmt", "yuv420p",
            "-r", str(TARGET_FPS),
            "-movflags", "+faststart",
            str(local_output),
        ]
        result = run_command(encode_cmd)

    if result.returncode != 0 or not valid_existing_video(local_output):
        return False, f"Merge failed: {result.stderr[-1500:]}"

    safe_copy_to_drive(local_output, output_path)
    local_output.unlink(missing_ok=True)
    list_file.unlink(missing_ok=True)
    return True, f"merged {len(clip_paths)} clips"

def write_csv(path: Path, rows: list[dict[str, Any]], fieldnames: list[str]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", newline="", encoding="utf-8-sig") as handle:
        writer = csv.DictWriter(handle, fieldnames=fieldnames)
        writer.writeheader()
        for row in rows:
            writer.writerow({field: row.get(field, "") for field in fieldnames})

# ============================================================
# 4) MAIN PIPELINE
# ============================================================
def main() -> None:
    ensure_ffmpeg()

    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    WORK_DIR.mkdir(parents=True, exist_ok=True)

    print("=" * 78)
    print("CardioVis stage segmentation")
    print(f"Merged input : {MERGED_DIR}")
    print(f"Original input: {ORIGINAL_DIR}")
    print(f"Output       : {OUTPUT_DIR}")
    print("=" * 78)

    if not MERGED_DIR.exists():
        raise FileNotFoundError(f"Merged folder does not exist: {MERGED_DIR}")

    # Save the exact manifest used for reproducibility.
    manifest_csv_rows = []
    for row in MANIFEST:
        start_sec = parse_doc_time(row["start"])
        end_sec = parse_doc_time(row["end"])
        manifest_csv_rows.append({
            **row,
            "start_hhmmss": seconds_to_hhmmss(start_sec),
            "end_hhmmss": seconds_to_hhmmss(end_sec),
            "duration_seconds": int(end_sec - start_sec),
        })

    write_csv(
        OUTPUT_DIR / "segment_manifest_used.csv",
        manifest_csv_rows,
        [
            "procedure", "patient", "stage", "source", "video_no",
            "start", "end", "start_hhmmss", "end_hhmmss", "duration_seconds",
        ],
    )
    write_csv(
        OUTPUT_DIR / "review_required.csv",
        REVIEW_REQUIRED,
        ["procedure", "patient", "stage", "issue"],
    )

    print("Indexing video files...")
    merged_index = video_files(MERGED_DIR)
    original_index = video_files(ORIGINAL_DIR)
    print(f"Found {len(merged_index)} merged-folder video files.")
    print(f"Found {len(original_index)} original-folder video files.")

    sorted_manifest = sorted(
        MANIFEST,
        key=lambda r: (
            PROCEDURE_ORDER[r["procedure"]],
            r["patient"],
            STAGE_ORDER[r["stage"]],
        ),
    )

    results: list[dict[str, Any]] = []
    successful: list[dict[str, Any]] = []

    total = len(sorted_manifest)
    for index, row in enumerate(sorted_manifest, start=1):
        procedure = row["procedure"]
        patient = int(row["patient"])
        stage = row["stage"]
        source_mode = row["source"]
        video_no = row["video_no"]
        start_seconds = parse_doc_time(row["start"])
        end_seconds = parse_doc_time(row["end"])

        source_tag = "merged" if source_mode == "merged" else f"video{video_no}"
        filename = (
            f"{procedure}_P{patient:02d}_{source_tag}_"
            f"{STAGE_SLUG[stage]}_"
            f"{time_slug(row['start'])}-{time_slug(row['end'])}.mp4"
        )
        output_path = OUTPUT_DIR / "segments" / procedure / STAGE_SLUG[stage] / filename

        base_result = {
            "procedure": procedure,
            "patient": patient,
            "stage": stage,
            "source_mode": source_mode,
            "video_no": video_no if video_no is not None else "",
            "start_document": row["start"],
            "end_document": row["end"],
            "start_hhmmss": seconds_to_hhmmss(start_seconds),
            "end_hhmmss": seconds_to_hhmmss(end_seconds),
            "duration_seconds": int(end_seconds - start_seconds),
            "source_file": "",
            "output_file": str(output_path),
            "status": "",
            "message": "",
        }

        print(
            f"[{index:02d}/{total}] {procedure} P{patient:02d} "
            f"{stage} {base_result['start_hhmmss']}->{base_result['end_hhmmss']}"
        )

        if valid_existing_video(output_path) and not OVERWRITE_SEGMENTS:
            base_result["status"] = "existing"
            base_result["message"] = "Segment already exists; skipped"
            results.append(base_result)
            successful.append({**row, "output_path": output_path})
            print("  [SKIP] Existing segment")
            continue

        if source_mode == "merged":
            source_path, resolution_message = resolve_merged_file(
                procedure, patient, merged_index
            )
        else:
            source_path, resolution_message = resolve_original_file(
                procedure, patient, int(video_no), original_index
            )

        if source_path is None:
            base_result["status"] = "missing_source"
            base_result["message"] = resolution_message
            results.append(base_result)
            print(f"  [MISSING] {resolution_message}")
            continue

        base_result["source_file"] = str(source_path)

        if DRY_RUN:
            base_result["status"] = "ready"
            base_result["message"] = resolution_message
            results.append(base_result)
            print(f"  [READY] {source_path}")
            continue

        ok, message = cut_segment(
            source_path=source_path,
            output_path=output_path,
            start_seconds=start_seconds,
            end_seconds=end_seconds,
        )

        base_result["status"] = "created" if ok else "failed"
        base_result["message"] = f"{resolution_message}; {message}"
        results.append(base_result)

        if ok:
            successful.append({**row, "output_path": output_path})
            print(f"  [OK] {output_path.name}")
        else:
            print(f"  [FAILED] {message}")

    report_path = OUTPUT_DIR / "segment_report.csv"
    write_csv(
        report_path,
        results,
        [
            "procedure", "patient", "stage", "source_mode", "video_no",
            "start_document", "end_document", "start_hhmmss", "end_hhmmss",
            "duration_seconds", "source_file", "output_file", "status", "message",
        ],
    )

    if DRY_RUN:
        status_counts: dict[str, int] = defaultdict(int)
        for result_row in results:
            status_counts[result_row["status"]] += 1
        print("\n" + "=" * 78)
        print("DRY RUN COMPLETE — no video was cut or merged")
        print("=" * 78)
        print(f"Ready rows             : {status_counts['ready']}")
        print(f"Existing output rows   : {status_counts['existing']}")
        print(f"Missing source rows    : {status_counts['missing_source']}")
        print(f"Report                 : {report_path}")
        print("Set DRY_RUN = False and run again after resolving missing/ambiguous sources.")
        print("=" * 78)
        return

    # ------------------------------------------------------------
    # Merge A: separate procedure + stage
    # ------------------------------------------------------------
    merge_results: list[dict[str, Any]] = []
    grouped_by_procedure_stage: dict[tuple[str, str], list[dict[str, Any]]] = defaultdict(list)
    for row in successful:
        grouped_by_procedure_stage[(row["procedure"], row["stage"])].append(row)

    print("\n" + "=" * 78)
    print("Merging clips by procedure and stage")
    print("=" * 78)

    for procedure in ("replacement", "repair"):
        for stage in ("root_needle", "atriotomy", "phrenic_nerve"):
            rows = sorted(
                grouped_by_procedure_stage.get((procedure, stage), []),
                key=lambda r: r["patient"],
            )
            clips = [Path(r["output_path"]) for r in rows]
            output_path = (
                OUTPUT_DIR
                / "merged_by_procedure_and_stage"
                / f"{procedure}_{STAGE_SLUG[stage]}.mp4"
            )
            ok, message = concatenate_clips(
                clips, output_path, overwrite=OVERWRITE_MERGES
            )
            merge_results.append({
                "scope": "procedure_and_stage",
                "procedure": procedure,
                "stage": stage,
                "clip_count": len(clips),
                "output_file": str(output_path),
                "status": "ok" if ok else "failed",
                "message": message,
            })
            print(f"{procedure:11s} | {stage:15s} | {len(clips):2d} clips | {message}")

    # ------------------------------------------------------------
    # Merge B: all procedures together, one video per stage
    # ------------------------------------------------------------
    grouped_by_stage: dict[str, list[dict[str, Any]]] = defaultdict(list)
    for row in successful:
        grouped_by_stage[row["stage"]].append(row)

    print("\n" + "=" * 78)
    print("Merging all procedures together, one output per stage")
    print("=" * 78)

    for stage in ("root_needle", "atriotomy", "phrenic_nerve"):
        rows = sorted(
            grouped_by_stage.get(stage, []),
            key=lambda r: (
                PROCEDURE_ORDER[r["procedure"]],
                r["patient"],
            ),
        )
        clips = [Path(r["output_path"]) for r in rows]
        output_path = (
            OUTPUT_DIR
            / "merged_by_stage_all_procedures"
            / f"ALL_{STAGE_SLUG[stage]}.mp4"
        )
        ok, message = concatenate_clips(
            clips, output_path, overwrite=OVERWRITE_MERGES
        )
        merge_results.append({
            "scope": "all_procedures_by_stage",
            "procedure": "all",
            "stage": stage,
            "clip_count": len(clips),
            "output_file": str(output_path),
            "status": "ok" if ok else "failed",
            "message": message,
        })
        print(f"ALL         | {stage:15s} | {len(clips):2d} clips | {message}")

    write_csv(
        OUTPUT_DIR / "merge_report.csv",
        merge_results,
        [
            "scope", "procedure", "stage", "clip_count",
            "output_file", "status", "message",
        ],
    )

    # ------------------------------------------------------------
    # Summary
    # ------------------------------------------------------------
    status_counts: dict[str, int] = defaultdict(int)
    for row in results:
        status_counts[row["status"]] += 1

    print("\n" + "=" * 78)
    print("DONE")
    print("=" * 78)
    print(f"Manifest rows             : {len(MANIFEST)}")
    print(f"Created segments          : {status_counts['created']}")
    print(f"Existing segments         : {status_counts['existing']}")
    print(f"Missing source segments   : {status_counts['missing_source']}")
    print(f"Failed segments           : {status_counts['failed']}")
    print(f"Manual review items       : {len(REVIEW_REQUIRED)}")
    print(f"Segment report            : {report_path}")
    print(f"Merge report              : {OUTPUT_DIR / 'merge_report.csv'}")
    print(f"Manual review report      : {OUTPUT_DIR / 'review_required.csv'}")
    print(f"Final merged videos folder: {OUTPUT_DIR}")
    print("=" * 78)

if __name__ == "__main__":
    main()


## Run the new Drive pipeline (FPS=20 + sample 1/20)

Cell below runs `colab_drive_stage_pipeline.py` with environment overrides.

- It trims and merges by stage directly on Drive.
- Saves final videos to `Cardiovis-related/stage_outputs/final`.
- Extracts frames at `FPS=20` to stage folders.
- Creates sample set (first frame in each block of 20 frames).
- Copies sample set to local runtime only if memory is sufficient.
- Prints live step-by-step progress in notebook output.
- Writes live progress snapshot to: `Cardiovis-related/stage_outputs/reports/run_progress.json`.


In [ ]:
# Update these paths if your Drive layout differs.
import os
from pathlib import Path

SCRIPT_PATH = Path('/content/drive/MyDrive/Cardiovis-related/colab_drive_stage_pipeline.py')

# You can still set explicit paths. If a path is wrong/missing, AUTODETECT_DRIVE_PATHS=1
# lets the script search under MyDrive + Shareddrives by folder/file name.
os.environ['DOCX_PATH'] = '/content/drive/MyDrive/passio-lab-1.docx'
os.environ['MERGED_DIR'] = '/content/drive/MyDrive/CardioVis-merged-videos'
os.environ['ROOT_VIDEOS_DIR'] = '/content/drive/MyDrive/3) Videos'
os.environ['CARDIOVIS_RELATED_DIR'] = '/content/drive/MyDrive/Cardiovis-related'
os.environ['AUTODETECT_DRIVE_PATHS'] = '1'

# Execution options
os.environ['DRY_RUN'] = '0'
os.environ['EXTRACT_FPS'] = '20'
os.environ['SAMPLE_EVERY_N_FRAMES'] = '20'
os.environ['DELETE_INTERMEDIATE_AFTER_MERGE'] = '1'
os.environ['TRY_LOCAL_DOWNLOAD'] = '1'
os.environ['LOCAL_MIN_FREE_GB'] = '5'
os.environ['LOCAL_SAFETY_MULTIPLIER'] = '1.25'
os.environ['TRIGGER_BROWSER_DOWNLOAD'] = '0'  # set to 1 if you want browser downloads

# Strict/rerun controls
os.environ['STRICT_AMBIGUOUS_GOC'] = '1'          # 1: ambiguous goc mapping will be skipped (strict)
os.environ['RERUN_SOURCE_NOT_FOUND_ONLY'] = '0'   # 1: rerun only cases previously skipped as source_not_found
# Optional CSV for manual goc mapping overrides with columns: patient,video_index,file_path
# os.environ['MANUAL_GOC_OVERRIDES_CSV'] = '/content/drive/MyDrive/CARDIOVIS/manual_goc_overrides.csv'

if not SCRIPT_PATH.exists():
    raise FileNotFoundError(f'Pipeline script not found: {SCRIPT_PATH}')

!python "$SCRIPT_PATH"


## Force Download All Sample Frames (Zip)

If Drive UI download fails, run the next cell.
It zips the whole `frame_samples_1_per_20` folder and triggers direct browser download from Colab.


In [ ]:
from pathlib import Path
import shutil
from google.colab import files

samples_dir = Path('/content/drive/MyDrive/Cardiovis-related/stage_outputs/frame_samples_1_per_20')
zip_base = Path('/content/frame_samples_1_per_20_all')
zip_path = zip_base.with_suffix('.zip')

if not samples_dir.exists():
    raise FileNotFoundError(f'Samples folder not found: {samples_dir}')

# Recreate zip cleanly
if zip_path.exists():
    zip_path.unlink()

print('Zipping folder...')
archive_path = shutil.make_archive(str(zip_base), 'zip', root_dir=str(samples_dir))
archive_path = Path(archive_path)
size_mb = archive_path.stat().st_size / (1024 * 1024)
print(f'Zip ready: {archive_path} ({size_mb:.2f} MB)')
print('Starting browser download...')
files.download(str(archive_path))
